In [ ]:
# B1: ResNet-50 Supervised Baseline

**Leaf-JEPA IRP** | Stage 3 — Baseline Establishment

## Rationale

B1 establishes the **supervised CNN performance floor**. ResNet-50 pretrained on ImageNet
is the standard baseline in agricultural ML literature. It answers the lower-bound question
for **RQ5**: can Leaf-JEPA + PEFT outperform a fully supervised CNN when labels are scarce (<=10%)?

This notebook also generates **out-of-fold (OOF) softmax probabilities** via 5-fold
cross-validation, which B6 uses for CleanLab label noise detection.

**Key metric**: Macro-F1 (not accuracy) due to 36.2x class imbalance in PlantVillage.

## Initialization

In [4]:
# Setup and hyperparameter documentation
import sys, json, time, copy, os
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader
import torchvision.models as tv_models


PROJECT_ROOT = Path("..").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))
print(PROJECT_ROOT)

from stage3_baseline_establishment.config_stage3 import *
from stage3_baseline_establishment.baseline_utils import (
    seed_everything, PlantVillageDataset, load_split,
    train_one_epoch, evaluate, EarlyStopping, save_results,
    plot_confusion_matrix, label_efficiency_sweep, get_oof_probabilities
)
from stage2_dataset_preparation.outputs.augmentation.transforms import (
    get_pretrain_transform, get_eval_transform, get_finetune_transform
)

verify_config()
seed_everything(RANDOM_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ── Save hyperparameters BEFORE training ──
hyperparams = {
    "baseline": "B1",
    "model": "ResNet-50 (torchvision, ImageNet pretrained)",
    "total_params": "25.6M",
    "optimiser": "AdamW",
    "backbone_lr": CNN_LR,
    "head_lr": CNN_HEAD_LR,
    "weight_decay": CNN_WEIGHT_DECAY,
    "batch_size": CNN_BATCH_SIZE,
    "max_epochs": CNN_EPOCHS,
    "patience": CNN_PATIENCE,
    "criterion": "CrossEntropyLoss",
    "scheduler": "CosineAnnealingLR",
    "amp": True,
    "norm_mean": NORM_MEAN,
    "norm_std": NORM_STD,
    "image_crop": IMAGE_CROP,
    "seeds": SUBSET_SEEDS,
    "fractions": LABEL_FRACTIONS,
}
BASELINES_DIR.mkdir(parents=True, exist_ok=True)
save_results(hyperparams, BASELINES_DIR / "B1_hyperparams.json")

/workspace/leaf-jepa
  ✅  All config checks passed.
Device: cuda
  Saved: /workspace/leaf-jepa/stage3_baseline_establishment/outputs/baselines/B1_hyperparams.json


## Full training (100% labels)

In [2]:
import wandb

# ── Load Stage 2 splits ──
train_paths, train_labels, class_names = load_split(SPLITS_DIR / "train_split.json")
val_paths, val_labels, _ = load_split(SPLITS_DIR / "val_split.json")
test_paths, test_labels, _ = load_split(SPLITS_DIR / "test_split.json")

print(f"Train: {len(train_paths):,}  Val: {len(val_paths):,}  Test: {len(test_paths):,}")
print(f"Classes: {len(class_names)}")

train_tf = get_pretrain_transform()
eval_tf  = get_eval_transform()

train_ds = PlantVillageDataset(train_paths, train_labels, transform=train_tf)
val_ds   = PlantVillageDataset(val_paths, val_labels, transform=eval_tf)
test_ds  = PlantVillageDataset(test_paths, test_labels, transform=eval_tf)

train_loader = DataLoader(train_ds, batch_size=CNN_BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=CNN_BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=CNN_BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)

# ── Build ResNet-50 ──
def build_resnet50():
    model = tv_models.resnet50(weights=tv_models.ResNet50_Weights.DEFAULT)
    model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
    nn.init.xavier_uniform_(model.fc.weight)
    nn.init.zeros_(model.fc.bias)

    backbone_params = [p for n, p in model.named_parameters() if "fc" not in n]
    head_params = list(model.fc.parameters())

    param_groups = [
        {"params": backbone_params, "lr": CNN_LR},
        {"params": head_params, "lr": CNN_HEAD_LR},
    ]
    return model, param_groups

# ── Train on 100% labels ──
seed_everything(RANDOM_SEED)
model, param_groups = build_resnet50()
model = model.to(device)

n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nResNet-50 trainable params: {n_trainable:,}")

criterion = nn.CrossEntropyLoss()
optimiser = torch.optim.AdamW(param_groups, weight_decay=CNN_WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=CNN_EPOCHS)
scaler = GradScaler(enabled=torch.cuda.is_available())
es = EarlyStopping(patience=CNN_PATIENCE)

if WANDB_ENTITY:
    os.environ["WANDB__SERVICE_WAIT"] = "10"
    os.environ["WANDB_DISABLED"] = "true"
    try:
        wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY,
                   name="B1-ResNet50-full",
                   config=hyperparams)
    except Exception:
        print("WandB init failed — training without logging")
        WANDB_ENTITY = False


print("\nTraining B1 ResNet-50 (100% labels)...")
t0 = time.time()
for epoch in range(CNN_EPOCHS):
    train_loss = train_one_epoch(model, train_loader, criterion,
                                 optimiser, scaler, device)
    val_result = evaluate(model, val_loader, device)
    scheduler.step()

    if WANDB_ENTITY:
        wandb.log({"train_loss": train_loss,
                    "val_macro_f1": val_result["macro_f1"],
                    "val_accuracy": val_result["accuracy"], "epoch": epoch})

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"  Epoch {epoch+1:3d} | loss={train_loss:.4f} | "
              f"val_F1={val_result['macro_f1']:.4f} | val_acc={val_result['accuracy']:.4f}")

    es.step(val_result["macro_f1"], model)
    if es.stopped:
        print(f"  Early stop at epoch {epoch+1}, best val F1: {es.best_score:.4f}")
        break

elapsed = time.time() - t0
es.load_best(model)
print(f"  Training time: {elapsed:.0f}s")

# ── Evaluate on test set ──
test_result = evaluate(model, test_loader, device)
print(f"\n  B1 Test macro-F1:  {test_result['macro_f1']:.4f}")
print(f"  B1 Test accuracy:  {test_result['accuracy']:.4f}")

if WANDB_ENTITY:
    wandb.log({"test_macro_f1": test_result["macro_f1"],
               "test_accuracy": test_result["accuracy"]})
    wandb.finish()

# Save aggregate results
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
b1_aggregate = {
    "baseline": "B1",
    "model": "ResNet-50",
    "macro_f1": test_result["macro_f1"],
    "accuracy": test_result["accuracy"],
    "per_class_f1": test_result["per_class_f1"],
    "training_time_s": elapsed,
    "best_val_f1": float(es.best_score),
    "trainable_params": n_trainable,
}
save_results(b1_aggregate, BASELINES_DIR / "B1_aggregate.json")

plot_confusion_matrix(
    np.array(test_result["confusion_matrix"]),
    class_names,
    FIGURES_DIR / f"B1_confusion_matrix_seed{RANDOM_SEED}.png",
    title=f"B1 ResNet-50 | F1={test_result['macro_f1']:.3f}"
)

Train: 38,013  Val: 8,146  Test: 8,146
Classes: 38


/tmp/ipykernel_13558/2351795243.py:52: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=torch.cuda.is_available())
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.



ResNet-50 trainable params: 23,585,894


wandb: Currently logged in as: muh-haleef02 (muh-haleef02-inform) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


WandB init failed — training without logging

Training B1 ResNet-50 (100% labels)...


...

  Epoch  15 | loss=0.0404 | val_F1=0.9891 | val_acc=0.9929


  Epoch  20 | loss=0.0308 | val_F1=0.9919 | val_acc=0.9953


  Epoch  25 | loss=0.0261 | val_F1=0.9948 | val_acc=0.9967


  Early stop at epoch 29, best val F1: 0.9964
  Training time: 3611s



  B1 Test macro-F1:  0.9951
  B1 Test accuracy:  0.9964
  Saved: /workspace/leaf-jepa/stage3_baseline_establishment/outputs/baselines/B1_aggregate.json
  Saved: /workspace/leaf-jepa/stage3_baseline_establishment/outputs/figures/B1_confusion_matrix_seed42.png


## OOF probabilities for CleanLab (5-fold CV)

In [ ]:
# OOF probabilities for CleanLab (5-fold CV)
from sklearn.model_selection import StratifiedKFold

print("\nGenerating OOF probabilities for CleanLab (5-fold CV)...")

all_train_paths = np.array(train_paths)
all_train_labels = np.array(train_labels)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
oof_probs = np.zeros((len(all_train_labels), NUM_CLASSES), dtype=np.float32)

for fold, (train_idx, val_idx) in enumerate(skf.split(all_train_paths, all_train_labels)):
    print(f"\n  Fold {fold+1}/5 ({len(train_idx)} train, {len(val_idx)} val)")
    seed_everything(RANDOM_SEED + fold)

    fold_train_ds = PlantVillageDataset(
        all_train_paths[train_idx].tolist(),
        all_train_labels[train_idx].tolist(),
        transform=train_tf
    )
    fold_val_ds = PlantVillageDataset(
        all_train_paths[val_idx].tolist(),
        all_train_labels[val_idx].tolist(),
        transform=eval_tf
    )

    fold_train_loader = DataLoader(fold_train_ds, batch_size=CNN_BATCH_SIZE,
                                   shuffle=True, num_workers=4, pin_memory=True)
    fold_val_loader = DataLoader(fold_val_ds, batch_size=CNN_BATCH_SIZE,
                                 shuffle=False, num_workers=4, pin_memory=True)

    fold_model, fold_pg = build_resnet50()
    fold_model = fold_model.to(device)
    fold_crit = nn.CrossEntropyLoss()
    fold_opt = torch.optim.AdamW(fold_pg, weight_decay=CNN_WEIGHT_DECAY)
    fold_sched = torch.optim.lr_scheduler.CosineAnnealingLR(fold_opt, T_max=CNN_EPOCHS)
    fold_scaler = GradScaler(enabled=torch.cuda.is_available())
    fold_es = EarlyStopping(patience=CNN_PATIENCE)

    for epoch in range(CNN_EPOCHS):
        train_one_epoch(fold_model, fold_train_loader, fold_crit,
                        fold_opt, fold_scaler, device)
        fold_val_result = evaluate(fold_model, fold_val_loader, device)
        fold_sched.step()
        fold_es.step(fold_val_result["macro_f1"], fold_model)
        if fold_es.stopped:
            break

    fold_es.load_best(fold_model)
    fold_probs = get_oof_probabilities(fold_model, fold_val_loader, device)
    oof_probs[val_idx] = fold_probs
    print(f"    Fold {fold+1} val F1: {fold_es.best_score:.4f}")

    del fold_model, fold_opt, fold_sched, fold_scaler
    torch.cuda.empty_cache()

# Save OOF data
np.save(BASELINES_DIR / "B1_oof_probabilities.npy", oof_probs)
np.save(BASELINES_DIR / "B1_oof_labels.npy", all_train_labels)
print(f"\n  OOF probs shape: {oof_probs.shape}")
print(f"  Saved to {BASELINES_DIR}/B1_oof_probabilities.npy")


Generating OOF probabilities for CleanLab (5-fold CV)...

  Fold 1/5 (30410 train, 7603 val)


/tmp/ipykernel_13558/1502949094.py:37: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  fold_scaler = GradScaler(enabled=torch.cuda.is_available())


    Fold 1 val F1: 0.9938

  Fold 2/5 (30410 train, 7603 val)


/tmp/ipykernel_13558/1502949094.py:37: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  fold_scaler = GradScaler(enabled=torch.cuda.is_available())


    Fold 2 val F1: 0.9935

  Fold 3/5 (30410 train, 7603 val)


/tmp/ipykernel_13558/1502949094.py:37: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  fold_scaler = GradScaler(enabled=torch.cuda.is_available())


    Fold 3 val F1: 0.9959

  Fold 4/5 (30411 train, 7602 val)


/tmp/ipykernel_13558/1502949094.py:37: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  fold_scaler = GradScaler(enabled=torch.cuda.is_available())


    Fold 4 val F1: 0.9947

  Fold 5/5 (30411 train, 7602 val)


/tmp/ipykernel_13558/1502949094.py:37: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  fold_scaler = GradScaler(enabled=torch.cuda.is_available())
  Training:  40%|███████████▉                  | 190/476 [00:34<00:52,  5.43it/s, loss=0.1777]

## Label efficiency sweep

In [3]:
print("\n" + "=" * 60)
print("  B1 LABEL EFFICIENCY SWEEP")
print("=" * 60)


sweep_results = label_efficiency_sweep(
    model_factory=build_resnet50,
    splits_dir=SPLITS_DIR,
    val_paths=val_paths, val_labels=val_labels,
    test_paths=test_paths, test_labels=test_labels,
    class_names=class_names,
    fractions=[f for f in LABEL_FRACTIONS if f < 1.0],  # skip 1.0, already done
    seeds=SUBSET_SEEDS,
    norm_mean=NORM_MEAN, norm_std=NORM_STD,
    batch_size=CNN_BATCH_SIZE,
    lr=CNN_LR, head_lr=CNN_HEAD_LR,
    weight_decay=CNN_WEIGHT_DECAY,
    epochs=CNN_EPOCHS, patience=CNN_PATIENCE,
    device=device,
    baseline_id="B1",
    baselines_dir=BASELINES_DIR,
    figures_dir=FIGURES_DIR,
    wandb_project=WANDB_PROJECT if WANDB_ENTITY else None,
    wandb_entity=WANDB_ENTITY,
)

print("\nB1 label efficiency sweep complete.")


  B1 LABEL EFFICIENCY SWEEP

  B1-frac0.01-seed42
  Trainable params: 23,585,894
  Training samples: 380


D:\IRP\leaf-jepa\stage3_baseline_establishment\baseline_utils.py:490: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=torch.cuda.is_available())


WandB init failed — training without logging


  Test macro-F1: 0.1448
  Test accuracy: 0.3817
  Saved: D:\IRP\leaf-jepa\stage3_baseline_establishment\outputs\figures\B1_confusion_matrix_frac0.01_seed42.png

  B1-frac0.01-seed123
  Trainable params: 23,585,894
  Training samples: 380


D:\IRP\leaf-jepa\stage3_baseline_establishment\baseline_utils.py:490: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=torch.cuda.is_available())


...

  Test macro-F1: 0.9720
  Test accuracy: 0.9773
  Saved: D:\IRP\leaf-jepa\stage3_baseline_establishment\outputs\figures\B1_confusion_matrix_frac0.50_seed42.png

  B1-frac0.50-seed123
  Trainable params: 23,585,894
  Training samples: 19,006


D:\IRP\leaf-jepa\stage3_baseline_establishment\baseline_utils.py:490: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=torch.cuda.is_available())


  Test macro-F1: 0.9763
  Test accuracy: 0.9811
  Saved: D:\IRP\leaf-jepa\stage3_baseline_establishment\outputs\figures\B1_confusion_matrix_frac0.50_seed123.png

  B1-frac0.50-seed456
  Trainable params: 23,585,894
  Training samples: 19,006


D:\IRP\leaf-jepa\stage3_baseline_establishment\baseline_utils.py:490: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=torch.cuda.is_available())


  Test macro-F1: 0.9718
  Test accuracy: 0.9767
  Saved: D:\IRP\leaf-jepa\stage3_baseline_establishment\outputs\figures\B1_confusion_matrix_frac0.50_seed456.png
  Saved: D:\IRP\leaf-jepa\stage3_baseline_establishment\outputs\baselines\B1_label_efficiency.json

B1 label efficiency sweep complete.


## Critical Analysis

B1 ResNet-50 provides the supervised CNN floor. Key observations to document:

1. **Full-data performance** — ResNet-50 at 100% labels should achieve 95-99% macro-F1 on PlantVillage, confirming the dataset is learnable with standard supervision.

2. **Label efficiency degradation** — How steeply does performance drop as labels decrease? If B1 drops sharply below 10% labels while PEFT methods (Stage 5) remain stable, this supports RQ5.

3. **Per-class F1 pattern** — Which diseases are hardest? Compare with Stage 2 inter-class similarity predictions. Consistently confused pairs validate the dataset analysis.

4. **OOF probabilities** — The 5-fold CV OOF predictions enable CleanLab in B6 to detect likely mislabelled samples. This closes the Stage 2 quality loop.

5. **Training time** — Record for the computational efficiency comparison in Stage 7.